# MEx 1 Prototype — 3-Layer CNN for MNIST (einops/einsum)

**Course:** AI231 — Machine Learning Operations

**Goal:** Build a 3-layer CNN where every layer/operation is implemented with `einops`/`einsum` (no `nn.Conv2d`/`nn.Linear`), train 5 epochs on GPU, report test accuracy, and visualize 16 test samples in a 4×4 grid with ground-truth vs. predicted labels.

**Architecture (all custom, einops/einsum):**
```
input (B, 1, 28, 28)
  → Conv2d 1×28×28×32  stride 1  (einsum im2col-style direct conv)
  → ReLU
  → MaxPool 2×2          (einops rearrange + reduce)
  → Conv2d 14×14×32→64   stride 1
  → ReLU
  → MaxPool 2×2          (einops rearrange + reduce)
  → Flatten              (einops rearrange)
  → Linear 7×7×64 → 128  (einsum matmul)
  → ReLU
  → Linear 128 → 10      (einsum matmul)
  → Softmax (manual, einsum-friendly)
```
Three weight-bearing layers: **Conv(32) → Conv(64) → FC(128)** (+ final FC head = the 4th linear, but the *CNN* is 3 layers deep as specified).

## 1. Setup & device selection

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch import nn
from einops import rearrange, reduce
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

# --- Device: pin to ONE GPU for MNIST (multi-GPU is overkill at this scale) ---
if torch.cuda.is_available():
    # Pick the GPU with the most free memory via nvidia-smi
    import subprocess
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=index,memory.free', '--format=csv,noheader,nounits']
    ).decode().strip().split('\n')
    idx = max(out, key=lambda l: int(l.split(',')[1].strip()))[0]
    os.environ['CUDA_VISIBLE_DEVICES'] = idx.strip()
    device = torch.device('cuda:0')  # remapped to the chosen physical GPU
    print(f"Using GPU {idx.strip()} ({torch.cuda.get_device_name(0)}), "
          f"free mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total")
else:
    device = torch.device('cpu')
    print("No CUDA — falling back to CPU")

torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {device}")

## 2. Data — MNIST load & normalize

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean/std
])

train_ds = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=transform)

BATCH = 128
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = torch.utils.data.DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} samples | Test: {len(test_ds)} samples")
print(f"Sample shape: {train_ds[0][0].shape}")

## 3. Custom einops/einsum layers

No `nn.Conv2d`, no `nn.Linear`. Each op is written explicitly with `einsum`/`rearrange`/`reduce`.

**Conv2d via einsum** — the direct (im2col-free) formulation:
$$y[b, h', w', o] = \sum_{i,j,c} W[i, j, c, o] \cdot x[b, h'+i, w'+j, c]$$
implemented as `torch.einsum('ijco,bhwc->bhwo', weight, padded_input)` after zero-padding.

**MaxPool via einops** — reshape spatial dims into pool windows, then `reduce(..., 'max')`.

**Linear via einsum** — `torch.einsum('bc,co->bo', x, weight) + bias`.

In [ ]:
class EinConv2d(nn.Module):
    """Conv2d written with einsum. Weight: (kh, kw, cin, cout)."""
    def __init__(self, cin, cout, k, stride=1, padding=0, bias=True):
        super().__init__()
        self.k, self.stride, self.padding = k, stride, padding
        self.weight = nn.Parameter(torch.empty(k, k, cin, cout))
        self.bias   = nn.Parameter(torch.zeros(cout)) if bias else None
        # Kaiming init (fan_in = k*k*cin)
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        if self.bias is not None:
            fan_in = k * k * cin
            nn.init.uniform_(self.bias, -1/np.sqrt(fan_in), 1/np.sqrt(fan_in))

    def forward(self, x):
        # x: (B, C, H, W)  ->  (B, H, W, C) for einsum convenience
        x = rearrange(x, 'b c h w -> b h w c')
        x = F.pad(x, (self.padding,)*4)  # pad H,W,C both sides (C padding harmless, sliced away)
        # Direct conv via einsum: slide kernel over input
        # Unfold input into patches: (B, H', W', k, k, C)
        B, H, W, C = x.shape
        H_out = (H - 2*self.padding - self.k) // self.stride + 1
        W_out = (W - 2*self.padding - self.k) // self.stride + 1
        patches = torch.stack([
            x[:, i:i+H_out*self.stride:self.stride, j:j+W_out*self.stride:self.stride, :, :]
            for i in range(self.k) for j in range(self.k)
        ], dim=-1)  # (B, H', W', k*k, C)
        patches = rearrange(patches, 'b h w kk c -> b h w (kk c)')
        w = rearrange(self.weight, 'i j c o -> (i j c) o')  # (k*k*C, Cout)
        y = torch.einsum('bhwd,do->bhwo', patches, w)  # (B, H', W', Cout)
        if self.bias is not None:
            y = y + self.bias
        return rearrange(y, 'b h w c -> b c h w')


class EinMaxPool(nn.Module):
    """2x2 max-pool via einops rearrange + reduce."""
    def __init__(self, kernel=2, stride=2):
        super().__init__()
        assert kernel == stride
        self.k = kernel

    def forward(self, x):
        # x: (B, C, H, W)
        x = rearrange(x, 'b c (h k1) (w k2) -> b c h w k1 k2', k1=self.k, k2=self.k)
        return reduce(x, 'b c h w k1 k2 -> b c h w', 'max')


class EinLinear(nn.Module):
    """Linear via einsum. Weight: (in, out)."""
    def __init__(self, fin, fout, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(fin, fout))
        self.bias   = nn.Parameter(torch.zeros(fout)) if bias else None
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -1/np.sqrt(fin), 1/np.sqrt(fin))

    def forward(self, x):
        # x: (..., fin)
        y = torch.einsum('...i,io->...o', x, self.weight)
        if self.bias is not None:
            y = y + self.bias
        return y


class EinSoftmax(nn.Module):
    """Numerically stable softmax (log-sum-exp trick)."""
    def forward(self, x):
        x_max = x.max(dim=-1, keepdim=True).values
        e = torch.exp(x - x_max)
        return e / e.sum(dim=-1, keepdim=True)


class ThreeLayerCNN(nn.Module):
    """
    3-layer CNN: Conv(32) -> Conv(64) -> FC(128), plus final FC head to 10 classes.
    Every op is einops/einsum-based.
    """
    def __init__(self):
        super().__init__()
        self.conv1 = EinConv2d(cin=1,  cout=32, k=3, padding=1)   # 28x28 -> 28x28
        self.pool1 = EinMaxPool(2, 2)                              # 28x28 -> 14x14
        self.conv2 = EinConv2d(cin=32, cout=64, k=3, padding=1)   # 14x14 -> 14x14
        self.pool2 = EinMaxPool(2, 2)                              # 14x14 -> 7x7
        self.fc1   = EinLinear(7*7*64, 128)                        # flatten -> 128
        self.fc2   = EinLinear(128, 10)                            # head
        self.softmax = EinSoftmax()

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = rearrange(x, 'b c h w -> b (c h w)')   # flatten
        x = torch.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits, self.softmax(logits)


model = ThreeLayerCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {n_params:,}")

## 4. Training — 5 epochs, AMP (fp16) on GPU

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
EPOCHS    = 5

history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits, probs = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * xb.size(0)
        correct += (probs.argmax(1) == yb).sum().item()
        total   += xb.size(0)

    train_acc = 100.0 * correct / total
    history['train_loss'].append(running_loss / total)
    history['train_acc'].append(train_acc)

    # Test accuracy
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            _, probs = model(xb)
            correct += (probs.argmax(1) == yb).sum().item()
            total   += xb.size(0)
    test_acc = 100.0 * correct / total
    history['test_acc'].append(test_acc)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={running_loss/total:.4f} | "
          f"train_acc={train_acc:.2f}% | test_acc={test_acc:.2f}%")

print(f"\nFINAL TEST ACCURACY: {history['test_acc'][-1]:.2f}%")

## 5. Results — 4×4 grid: image, ground truth, prediction

In [ ]:
# Grab 16 test samples (first 16 for reproducibility)
model.eval()
xb, yb = next(iter(test_dl))
xb, yb = xb[:16].to(device), yb[:16]
with torch.no_grad():
    _, probs = model(xb)
preds = probs.argmax(1).cpu().numpy()
truth = yb.numpy()
imgs  = xb.cpu().numpy()  # (16, 1, 28, 28), normalized

# Denormalize for display
imgs = imgs * 0.3081 + 0.1307
imgs = np.clip(imgs, 0, 1)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img, t, p in zip(axes.ravel(), imgs, truth, preds):
    ax.imshow(img[0], cmap='gray')
    color = 'green' if t == p else 'red'
    ax.set_title(f"GT: {t}  Pred: {p}", color=color, fontsize=11)
    ax.axis('off')
fig.suptitle(f"MNIST 4×4 Grid — Test Acc: {history['test_acc'][-1]:.2f}%  "
             f"(green = correct, red = wrong)", fontsize=13)
plt.tight_layout()
plt.show()

n_correct = int((preds == truth).sum())
print(f"Grid accuracy: {n_correct}/16 = {100*n_correct/16:.1f}%")

## 6. Training curves

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
epochs = range(1, EPOCHS + 1)
ax1.plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='tab:blue')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(epochs, history['train_acc'], 's--', label='Train Acc', color='tab:orange')
ax2.plot(epochs, history['test_acc'],  '^-',  label='Test Acc',  color='tab:green')
ax2.set_ylabel('Accuracy (%)', color='tab:green')
ax1.set_title('Training Progress (5 epochs)')
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='center left')
plt.tight_layout()
plt.show()